# HEOM with QuTiP Backend

This notebook demonstrates how to use [QuTiP](https://qutip.org/)'s HEOM solver
as a backend for quantarhei. The `QuTip_KTHierarchyPropagator` provides the same
interface as the native `KTHierarchyPropagator`, but delegates the actual
computation to QuTiP's optimised `HEOMSolver` with Padé bath decomposition.

**Advantages of the QuTiP backend:**
- Padé decomposition converges faster than Matsubara — fewer hierarchy levels needed
- Optimised ODE integrators (adaptive Runge-Kutta, DOP853, etc.)
- Access to QuTiP's progress reporting and options

**Requirements:** `pip install quantarhei[qutip]`

In [ ]:
%matplotlib inline
%config InlineBackend.figure_format = 'svg'

## 1. Define the model

We use the same strongly coupled dimer as in the native HEOM notebook
(notebook 05): two molecules with $J = 100$ cm$^{-1}$ coupling and an
overdamped Brownian bath ($\lambda = 100$ cm$^{-1}$, $\tau_c = 50$ fs, $T = 300$ K).

In [ ]:
import numpy as np

import quantarhei as qr

# Time axis: 500 steps of 1 fs = 500 fs total
timea = qr.TimeAxis(0.0, 500, 1.0)

# Two two-level molecules
with qr.energy_units("1/cm"):
    m1 = qr.Molecule([0.0, 10000.0])
    m2 = qr.Molecule([0.0, 10100.0])

# Build dimer aggregate with J = 100 cm-1
agg = qr.Aggregate([m1, m2])
with qr.energy_units("1/cm"):
    agg.set_resonance_coupling(0, 1, 100.0)

# Bath: overdamped Brownian oscillator
cpar = dict(
    ftype="OverdampedBrownian-HighTemperature",
    reorg=100,    # cm-1
    cortime=50,   # fs
    T=300,        # K
)
with qr.energy_units("1/cm"):
    cfce = qr.CorrelationFunction(timea, cpar)

m1.set_transition_environment((0, 1), cfce)
m2.set_transition_environment((0, 1), cfce)

agg.build()

ham = agg.get_Hamiltonian()
sbi = agg.get_SystemBathInteraction()

print("Hamiltonian (cm-1):")
with qr.energy_units("1/cm"):
    print(np.round(ham.data).astype(int))

## 2. Propagate with the QuTiP backend

The `QuTip_KTHierarchyPropagator` is a drop-in replacement for
`KTHierarchyPropagator`. It uses QuTiP's `DrudeLorentzPadeBath` for
faster convergence.

In [ ]:
# Build hierarchy
depth = 3
Hy = qr.KTHierarchy(ham, sbi, depth)
print(f"Hierarchy depth {Hy.depth} — auxiliary matrices: {Hy.hsize}")

# Initial condition: all population on site 2
rhoi = qr.ReducedDensityMatrix(dim=ham.dim)
rhoi.data[2, 2] = 1.0

# QuTiP-backed propagator
kprop_qt = qr.QuTip_KTHierarchyPropagator(timea, Hy)
rhot_qt = kprop_qt.propagate(rhoi, report_hierarchy=False)
print("QuTiP HEOM propagation done")

## 3. Compare with native implementation

We run the same calculation with the native quantarhei HEOM solver
and overlay the results.

In [ ]:
# Native propagator (same hierarchy)
kprop_native = qr.KTHierarchyPropagator(timea, Hy)
rhot_native = kprop_native.propagate(rhoi, report_hierarchy=False)
print("Native HEOM propagation done")

In [ ]:
import matplotlib.pyplot as plt

N = timea.length
t = timea.data

fig, ax = plt.subplots(figsize=(8, 5))

# Native (solid lines)
ax.plot(t, np.real(rhot_native.data[:N, 1, 1]), 'b-', label='Site 1 (native)', lw=2)
ax.plot(t, np.real(rhot_native.data[:N, 2, 2]), 'r-', label='Site 2 (native)', lw=2)

# QuTiP (dashed lines)
ax.plot(t, np.real(rhot_qt.data[:N, 1, 1]), 'b--', label='Site 1 (QuTiP)', lw=2)
ax.plot(t, np.real(rhot_qt.data[:N, 2, 2]), 'r--', label='Site 2 (QuTiP)', lw=2)

ax.set_xlabel('Time [fs]')
ax.set_ylabel(r'Population $\rho_{ii}$')
ax.set_title('HEOM: Native vs QuTiP backend (depth = 3)')
ax.legend()
ax.set_xlim(0, 500)
plt.tight_layout()
plt.show()

The dashed (QuTiP) and solid (native) lines should overlap closely.
Small differences arise from different ODE integrators and the Padé vs
Matsubara bath decomposition — both converge to the exact result as
hierarchy depth increases.

## 4. Convergence advantage of Padé decomposition

The QuTiP backend uses the Padé decomposition (`DrudeLorentzPadeBath`)
instead of the Matsubara expansion. For the same number of exponential
terms $N_k$, Padé converges faster. This means you can use a **shallower
hierarchy** (less memory, faster runtime) for the same accuracy.

In [ ]:
# Compare depth 2 (shallow) between native and QuTiP
Hy2 = qr.KTHierarchy(ham, sbi, 2)
print(f"Hierarchy depth {Hy2.depth} — auxiliary matrices: {Hy2.hsize}")

kprop_qt2 = qr.QuTip_KTHierarchyPropagator(timea, Hy2)
rhot_qt2 = kprop_qt2.propagate(rhoi, report_hierarchy=False)

kprop_native2 = qr.KTHierarchyPropagator(timea, Hy2)
rhot_native2 = kprop_native2.propagate(rhoi, report_hierarchy=False)

# Reference: native at depth 4
Hy4 = qr.KTHierarchy(ham, sbi, 4)
kprop_ref = qr.KTHierarchyPropagator(timea, Hy4)
rhot_ref = kprop_ref.propagate(rhoi, report_hierarchy=False)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(t, np.real(rhot_ref.data[:N, 2, 2]), 'k-', label='Reference (native, depth 4)', lw=2)
ax.plot(t, np.real(rhot_native2.data[:N, 2, 2]), 'r-', label='Native (depth 2)', lw=1.5)
ax.plot(t, np.real(rhot_qt2.data[:N, 2, 2]), 'b--', label='QuTiP Padé (depth 2)', lw=1.5)

ax.set_xlabel('Time [fs]')
ax.set_ylabel(r'$\rho_{22}(t)$')
ax.set_title('Convergence: Padé (QuTiP) vs Matsubara (native) at depth 2')
ax.legend()
ax.set_xlim(0, 500)
plt.tight_layout()
plt.show()

## Summary

| Feature | Native HEOM | QuTiP backend |
|---------|-------------|---------------|
| Bath decomposition | Matsubara | Padé (faster convergence) |
| ODE integrator | Fixed-step short-time expansion | Adaptive (vern9, dop853, etc.) |
| Dependency | None | `pip install quantarhei[qutip]` |
| API | `KTHierarchyPropagator` | `QuTip_KTHierarchyPropagator` |

Both produce the same physics — choose the QuTiP backend when you need
faster convergence or want to leverage QuTiP's solver options.